In [0]:
bronzedf = spark.table("databrickstraining.bronze.leanrers_bronze")

In [0]:
stagingdf = bronzedf.select(
    "learner_id",
    "learner_name",
    "last_name",
    "email",
    "phone",
    "created_time",
    "registered_date",
    "last_activity_time",
    "course_name",
    "total_fees",
    "fee_paid",
    "due_amount",
    "due_date"
)

In [0]:
from pyspark.sql.functions import count
stagingdf.agg(count("*")).show()

In [0]:
from pyspark.sql.functions import concat_ws,col
import regex


stagingdf = stagingdf.withColumnRenamed("learner_id", "id").withColumn("name", concat_ws("_", col("last_name"), col("learner_name"))).select("id", "name", "email", "phone", "created_time", "registered_date", "last_activity_time", "course_name", "total_fees", "fee_paid", "due_amount", "due_date").filter(~col("name").rlike("sample"))

In [0]:
stagingdf = stagingdf.fillna({"phone":"Not_available","email":"Not_available","created_time":"Not_available"})

In [0]:
stagingdf = stagingdf.filter(~col("created_time").isin(['Fullstack','Cloudops'])).filter(~col("registered_date").rlike("DL")).filter(col("id").isNotNull())

In [0]:
%sql
create database if not exists databrickstraining.staging

In [0]:
stagingdf.write.mode("Append").saveAsTable("databrickstraining.staging.learners_stage")

In [0]:
stagingdf.write.mode("Append").saveAsTable("databrickstraining.training8am.learners")